# Análise Financeira 2026 — Versão Final Profissional

Notebook exclusivamente analítico.

## Regras

- não recalcula regras financeiras;
- não recalcula Danone;
- não recalcula custo de estrutura;
- não recalcula ocupação;
- usa apenas os outputs oficiais do pipeline final;
- métricas operacionais usam `viagens_2026.parquet`;
- métricas financeiras de detalhe usam `inform_27_2026_final.parquet`;
- reconciliação Danone usa `validacao_danone_2026.csv`;
- Setembro pode existir no detalhe, mas a análise comparável pode ser limitada a Janeiro–Agosto.

## 00. Configuração

In [1]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import numpy as np
import pandas as pd

# ==========================
# 2. Parâmetros
# ==========================
ANO = 2026
MES_ANALISE_COMPLETA_MAX = 8

# ==========================
# 3. Caminhos
# ==========================
def get_output_path():
    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\Documents\python\01.Financeiro"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        )

    return Path.cwd()

OUTPUT = get_output_path()

# ==========================
# 4. Ficheiros oficiais
# ==========================
FICHEIROS = {
    "detalhe": OUTPUT / "inform_27_2026_final.parquet",
    "viagens": OUTPUT / "viagens_2026.parquet",
    "validacao": OUTPUT / "validacao_financeira_2026.csv",
    "danone": OUTPUT / "validacao_danone_2026.csv",
}

# ==========================
# 5. Validar ficheiros
# ==========================
for nome, caminho in FICHEIROS.items():
    if not caminho.exists():
        raise FileNotFoundError(
            f"Ficheiro em falta ({nome}): {caminho}"
        )

# ==========================
# 6. Ler outputs oficiais
# ==========================
df = pd.read_parquet(
    FICHEIROS["detalhe"]
)

viagens = pd.read_parquet(
    FICHEIROS["viagens"]
)

validacao = pd.read_csv(
    FICHEIROS["validacao"],
    sep=";",
    decimal=",",
)

validacao_danone = pd.read_csv(
    FICHEIROS["danone"],
    sep=";",
    decimal=",",
)

validacao

,controlo,valor,tolerancia,estado
0,Reconciliação Danone classificada,5.820766e-09,0.01,OK
1,Estrutura mensal,1.455192e-11,0.01,OK
2,Custo detalhe = viagens,0.000000e+00,0.01,OK
3,Ingresso detalhe = viagens,9.313226e-10,0.01,OK
4,CODEUT texto nan,0.000000e+00,0.00,OK
5,CODEUT único em viagens,0.000000e+00,0.00,OK
6,Ocupação OK > 100%,0.000000e+00,0.00,OK
7,Margem consistente,0.000000e+00,0.01,OK


## 01. Funções de análise

In [2]:
# ==========================
# 1. Divisão segura
# ==========================
def dividir_seguro(numerador, denominador):
    n = pd.to_numeric(
        numerador,
        errors="coerce",
    )

    d = pd.to_numeric(
        denominador,
        errors="coerce",
    )

    return n.div(
        d.where(d.ne(0))
    )

# ==========================
# 2. Filtro comparável
# ==========================
def filtrar_periodo_completo(dataframe, coluna_data):
    dados = dataframe.copy()

    data = pd.to_datetime(
        dados[coluna_data],
        errors="coerce",
    )

    return dados.loc[
        data.dt.year.eq(ANO)
        & data.dt.month.le(
            MES_ANALISE_COMPLETA_MAX
        )
    ].copy()

## 02. Estado dos dados

In [3]:
# ==========================
# 1. Validar pipeline
# ==========================
if not validacao["estado"].eq("OK").all():
    raise ValueError(
        "O pipeline contém validações com ERRO."
    )

# ==========================
# 2. Resumo de qualidade
# ==========================
estado_pipeline = pd.DataFrame({
    "metrica": [
        "Validações financeiras",
        "Linhas detalhe",
        "Viagens",
        "Período comparável até mês",
    ],
    "valor": [
        int(validacao["estado"].eq("OK").sum()),
        len(df),
        len(viagens),
        MES_ANALISE_COMPLETA_MAX,
    ],
})

estado_pipeline

,metrica,valor
0,Validações financeiras,8
1,Linhas detalhe,102291
2,Viagens,16156
3,Período comparável até mês,8


In [4]:
# ==========================
# 1. Reconciliação Danone
# ==========================
validacao_danone

,estado_reconciliacao,casos,referencias,ingresso_base,extra,ingresso
0,ASSOCIADO_EXACTO,13866,13866,1.050288e+06,152965.109168,1.203253e+06
1,EXCLUIDO_CODACT_13,24565,24564,4.130858e+04,5962.101594,4.727069e+04
2,FORA_INFORM27,2165,2165,5.491476e+03,779.558457,6.271034e+03
3,REFERENCIA_EXISTE_DATA_DIFERENTE,83,83,2.112202e+03,293.022721,2.405225e+03


## 03. Resumo financeiro mensal

In [5]:
# ==========================
# 1. Preparar datas
# ==========================
df["mes_estrutura"] = pd.to_datetime(
    df["mes_estrutura"],
    errors="coerce",
)

# ==========================
# 2. Agregar mês
# ==========================
resumo_mensal = (
    df.groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        linhas=("CODEUT", "size"),
        viagens=("CODEUT", "nunique"),
        paletes=("PALETS", "sum"),
        peso_kg=("PESO_BRUTO", "sum"),
        ingresso=("total_ingresso_c", "sum"),
        custo=("custo_total_c", "sum"),
        margem=("margem", "sum"),
        estrutura=("custo_estrutura", "sum"),
        combustivel=("acerto_combustivel", "sum"),
        ingresso_danone=("ingresso_danone", "sum"),
    )
)

# ==========================
# 3. Indicadores
# ==========================
resumo_mensal["peso_ton"] = (
    resumo_mensal["peso_kg"]
    / 1000
)

resumo_mensal["margem_pct"] = (
    dividir_seguro(
        resumo_mensal["margem"],
        resumo_mensal["ingresso"],
    )
    * 100
)

resumo_mensal["ingresso_por_ton"] = (
    dividir_seguro(
        resumo_mensal["ingresso"],
        resumo_mensal["peso_ton"],
    )
)

resumo_mensal["custo_por_ton"] = (
    dividir_seguro(
        resumo_mensal["custo"],
        resumo_mensal["peso_ton"],
    )
)

resumo_mensal

,mes_estrutura,linhas,viagens,paletes,peso_kg,ingresso,custo,margem,estrutura,combustivel,ingresso_danone,peso_ton,margem_pct,ingresso_por_ton,custo_por_ton
0,2026-01-01,11327,1923,46298.001,10473868.763,464356.71835,513737.91,-49381.19165,103000.0,0.000000,137235.99485,10473.868763,-10.634323,44.334785,49.049489
1,2026-02-01,10645,1718,38238.75,8952691.761,411041.180994,458867.4,-47826.219006,103000.0,2058.953390,127578.022104,8952.691761,-11.635384,45.91258,51.254685
2,2026-03-01,12281,1966,46487.563,12553858.055,508804.760301,540931.2,-32126.439699,103000.0,2926.743206,147235.779595,12553.858055,-6.3141,40.529753,43.088842
3,2026-04-01,12577,1972,48077.75,11656719.178,558217.411672,572365.89,-14148.478328,103000.0,30094.108710,148533.128462,11656.719178,-2.534582,47.888038,49.1018
4,2026-05-01,12581,1953,47682.05,11801330.019,564403.313304,573870.16,-9466.846696,103000.0,45637.587756,147257.889548,11801.330019,-1.677319,47.825399,48.627583
5,2026-06-01,12460,1966,45447.16,11952113.721,551492.711347,580716.09,-29223.378653,103000.0,40162.106093,157258.937254,11952.113721,-5.29896,46.141856,48.586895
6,2026-07-01,14179,2167,49278.5,13429174.515,651931.579208,643897.92,8033.659208,103000.0,33775.387012,178657.705196,13429.174515,1.232286,48.545916,47.947692
7,2026-08-01,13273,2019,49251.0,12551971.754,553292.0225,555267.22,-1975.1975,103000.0,32553.613206,159495.688794,12551.971754,-0.35699,44.080088,44.23745
8,2026-09-01,2968,472,10311.987,2805432.422,24456.199554,103000.0,-78543.800446,103000.0,2223.290869,22232.908686,2805.432422,-321.161104,8.717444,36.714483


## 04. Resumo financeiro comparável — Janeiro a Agosto

In [6]:
# ==========================
# 1. Filtrar período completo
# ==========================
df_comparavel = filtrar_periodo_completo(
    df,
    "FENTREGA",
)

# ==========================
# 2. KPIs
# ==========================
kpi_financeiro = pd.DataFrame({
    "metrica": [
        "Ingresso",
        "Custo",
        "Margem",
        "Margem %",
        "Paletes",
        "Peso ton",
        "Viagens",
    ],
    "valor": [
        df_comparavel[
            "total_ingresso_c"
        ].sum(),
        df_comparavel[
            "custo_total_c"
        ].sum(),
        df_comparavel[
            "margem"
        ].sum(),
        (
            df_comparavel[
                "margem"
            ].sum()
            / df_comparavel[
                "total_ingresso_c"
            ].sum()
            * 100
        )
        if df_comparavel[
            "total_ingresso_c"
        ].sum() != 0
        else np.nan,
        df_comparavel[
            "PALETS"
        ].sum(),
        df_comparavel[
            "PESO_BRUTO"
        ].sum() / 1000,
        df_comparavel[
            "CODEUT"
        ].nunique(),
    ],
})

kpi_financeiro

,metrica,valor
0,Ingresso,4.263540e+06
1,Custo,4.439654e+06
2,Margem,-1.761141e+05
3,Margem %,-4.130701e+00
4,Paletes,3.707608e+05
5,Peso ton,9.337173e+04
6,Viagens,1.568400e+04


## 05. KPIs oficiais de viagens

In [7]:
# ==========================
# 1. Preparar período
# ==========================
viagens["data"] = pd.to_datetime(
    viagens["data"],
    errors="coerce",
)

viagens_comparaveis = (
    filtrar_periodo_completo(
        viagens,
        "data",
    )
)

viagens_validas = (
    viagens_comparaveis.loc[
        viagens_comparaveis[
            "ocupacao_estado"
        ].eq("OK")
    ]
    .copy()
)

# ==========================
# 2. KPIs
# ==========================
kpi_viagens = pd.DataFrame({
    "metrica": [
        "Viagens",
        "Paletes",
        "Peso ton",
        "Ocupação média %",
        "Custo por viagem",
        "Custo por palete",
        "Custo por ton",
        "Ingresso por ton",
        "Margem",
        "Margem %",
    ],
    "valor": [
        viagens_comparaveis[
            "CODEUT"
        ].nunique(),
        viagens_comparaveis[
            "paletes"
        ].sum(),
        viagens_comparaveis[
            "peso_ton"
        ].sum(),
        viagens_validas[
            "ocupacao_pct"
        ].mean(),
        viagens_comparaveis[
            "custo_total"
        ].mean(),
        dividir_seguro(
            pd.Series([
                viagens_comparaveis[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                viagens_comparaveis[
                    "paletes"
                ].sum()
            ]),
        ).iloc[0],
        dividir_seguro(
            pd.Series([
                viagens_comparaveis[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                viagens_comparaveis[
                    "peso_ton"
                ].sum()
            ]),
        ).iloc[0],
        dividir_seguro(
            pd.Series([
                viagens_comparaveis[
                    "ingresso_total"
                ].sum()
            ]),
            pd.Series([
                viagens_comparaveis[
                    "peso_ton"
                ].sum()
            ]),
        ).iloc[0],
        viagens_comparaveis[
            "margem"
        ].sum(),
        dividir_seguro(
            pd.Series([
                viagens_comparaveis[
                    "margem"
                ].sum()
            ]),
            pd.Series([
                viagens_comparaveis[
                    "ingresso_total"
                ].sum()
            ]),
        ).iloc[0] * 100,
    ],
})

kpi_viagens

,metrica,valor
0,Viagens,15684.000000
1,Paletes,370760.774000
2,Peso ton,93371.727766
3,Ocupação média %,66.036864
4,Custo por viagem,283.068974
5,Custo por palete,11.974443
6,Custo por ton,47.548159
7,Ingresso por ton,45.661999
8,Margem,-176114.092326
9,Margem %,-4.130701


## 06. Transportadores

In [8]:
# ==========================
# 1. Agregar transportador
# ==========================
transportadores = (
    viagens_comparaveis.groupby(
        "TRANSPORTISTA",
        dropna=False,
        as_index=False,
    )
    .agg(
        viagens=("CODEUT", "nunique"),
        paletes=("paletes", "sum"),
        peso_ton=("peso_ton", "sum"),
        custo=("custo_total", "sum"),
        ingresso=("ingresso_total", "sum"),
        margem=("margem", "sum"),
        ocupacao_media=("ocupacao_pct", "mean"),
    )
)

# ==========================
# 2. Indicadores
# ==========================
transportadores["custo_por_viagem"] = (
    dividir_seguro(
        transportadores["custo"],
        transportadores["viagens"],
    )
)

transportadores["custo_por_palete"] = (
    dividir_seguro(
        transportadores["custo"],
        transportadores["paletes"],
    )
)

transportadores["custo_por_ton"] = (
    dividir_seguro(
        transportadores["custo"],
        transportadores["peso_ton"],
    )
)

transportadores["margem_pct"] = (
    dividir_seguro(
        transportadores["margem"],
        transportadores["ingresso"],
    )
    * 100
)

transportadores = (
    transportadores.sort_values(
        "custo",
        ascending=False,
    )
    .reset_index(drop=True)
)

transportadores

,TRANSPORTISTA,viagens,paletes,peso_ton,custo,ingresso,margem,ocupacao_media,custo_por_viagem,custo_por_palete,custo_por_ton,margem_pct
0,"TRANSPORTES FLORENCIO SILVA, LDA. -",6394,111902.341,22023.502953,1403144.799498,1073780.465245,-329364.334253,66.647139,219.447107,12.539012,63.711245,-30.67334
1,JMR PRESTAÇAO DE SERVIÇOS PARA A DISTRIBUÇAO S.A,1982,71691.25,19951.840536,608139.316138,647366.829346,39227.513209,77.605525,306.831138,8.482755,30.480362,6.059549
2,MODELO CONTINENTE HIPERMERCADOS S.A.,1322,42014.73,14449.395407,380718.795839,471576.759237,90857.963397,78.451734,287.986986,9.061555,26.348424,19.266845
3,CMTIR TRANSPORTES NAC. INTERN. S.A,953,9591.47,2307.711939,305754.00553,281235.548147,-24518.457383,44.904896,320.833164,31.8777,132.492275,-8.718122
4,TRANSPORTES PAULO COSTA & FERREIRA LDA,527,16876.0,2844.84946,268785.586538,248244.591856,-20540.994682,80.40404,510.029576,15.927091,94.48148,-8.274498
5,"TJA-TRANSPORTES J.AMARAL, S.A. -",813,19299.42,3507.697106,253279.993657,262386.603808,9106.610151,51.080898,311.537508,13.12371,72.206917,3.470684
6,TORRESTIR - TRANSPORTES NACIONAIS E INTERNACIO...,166,7880.0,603.768741,209564.218408,137945.658985,-71618.559423,88.232891,1262.435051,26.594444,347.093521,-51.917951
7,TRANSAURA TRANSPORTES LDA,820,22269.0,3188.060126,175404.520324,206046.257433,30641.737109,77.752763,213.907952,7.876623,55.019201,14.87129
8,"TRANSPORTES FIGUEIREDO & FIGUEIREDO, LDA",331,3805.0,1460.715201,138105.228759,109085.345928,-29019.882831,32.732374,417.236341,36.295724,94.546308,-26.602916
9,"TRANSPARENTODISSEIA - Transportes Unipessoal, ...",432,10211.0,6162.331945,136932.27602,179973.946748,43041.670727,49.408173,316.972861,13.410271,22.220854,23.915501


## 07. Actividades / CODACT

In [9]:
# ==========================
# 1. Agregar actividade
# ==========================
atividade = (
    df_comparavel.groupby(
        "CODACT",
        dropna=False,
        as_index=False,
    )
    .agg(
        linhas=("CODEUT", "size"),
        viagens=("CODEUT", "nunique"),
        paletes=("PALETS", "sum"),
        peso_kg=("PESO_BRUTO", "sum"),
        ingresso=("total_ingresso_c", "sum"),
        custo=("custo_total_c", "sum"),
        margem=("margem", "sum"),
    )
)

# ==========================
# 2. Indicadores
# ==========================
atividade["peso_ton"] = (
    atividade["peso_kg"]
    / 1000
)

atividade["margem_pct"] = (
    dividir_seguro(
        atividade["margem"],
        atividade["ingresso"],
    )
    * 100
)

atividade["ingresso_por_ton"] = (
    dividir_seguro(
        atividade["ingresso"],
        atividade["peso_ton"],
    )
)

atividade["custo_por_ton"] = (
    dividir_seguro(
        atividade["custo"],
        atividade["peso_ton"],
    )
)

atividade = (
    atividade.sort_values(
        "ingresso",
        ascending=False,
    )
    .reset_index(drop=True)
)

atividade

,CODACT,linhas,viagens,paletes,peso_kg,ingresso,custo,margem,peso_ton,margem_pct,ingresso_por_ton,custo_por_ton
0,11,18578,7345,125494.0,39379732.156,1262511.592342,1328572.337426,-66060.745084,39379.732156,-5.232486,32.059933,33.737465
1,74,19474,4191,36310.59,4135468.975,399564.840036,494524.637418,-94959.797382,4135.468975,-23.765804,96.618991,119.581271
2,429,7574,2248,12770.0,1820728.77,391175.197463,272144.602874,119030.594589,1820.72877,30.428973,214.845398,149.47015
3,118,1609,1405,3864.0,2093839.01,221987.099499,86948.014634,135039.084865,2093.83901,60.831952,106.019182,41.525645
4,260,3927,1832,18114.7,7428891.888,205461.68,198315.541148,7146.138852,7428.891888,3.478088,27.65711,26.695171
5,131,3901,2157,10111.0,1320115.538,165561.10375,100443.999807,65117.103943,1320.115538,39.331161,125.414101,76.087279
6,311,17934,4706,27947.0,8191828.36,162248.86,314070.765654,-151821.905654,8191.82836,-93.573481,19.806184,38.33952
7,415,1354,1087,7847.0,2031943.752,153878.236466,173461.03003,-19582.793564,2031.943752,-12.726162,75.729575,85.367043
8,551,246,244,8052.0,0.0,143891.0,144318.405262,-427.405262,0.0,-0.297034,<NA>,<NA>
9,211,1926,1552,6350.734,726090.0,118249.866,134200.929684,-15951.063684,726.09,-13.489287,162.858414,184.826853


## 08. Rotas

In [10]:
# ==========================
# 1. Agregar rota
# ==========================
rotas = (
    viagens_comparaveis.groupby(
        [
            "LOCORIGEN",
            "LOCDESTINO",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        viagens=("CODEUT", "nunique"),
        paletes=("paletes", "sum"),
        peso_ton=("peso_ton", "sum"),
        custo=("custo_total", "sum"),
        ingresso=("ingresso_total", "sum"),
        margem=("margem", "sum"),
        ocupacao_media=("ocupacao_pct", "mean"),
    )
)

# ==========================
# 2. Indicadores
# ==========================
rotas["custo_por_palete"] = (
    dividir_seguro(
        rotas["custo"],
        rotas["paletes"],
    )
)

rotas["custo_por_ton"] = (
    dividir_seguro(
        rotas["custo"],
        rotas["peso_ton"],
    )
)

rotas["margem_pct"] = (
    dividir_seguro(
        rotas["margem"],
        rotas["ingresso"],
    )
    * 100
)

rotas = (
    rotas.sort_values(
        "custo",
        ascending=False,
    )
    .reset_index(drop=True)
)

rotas

,LOCORIGEN,LOCDESTINO,viagens,paletes,peso_ton,custo,ingresso,margem,ocupacao_media,custo_por_palete,custo_por_ton,margem_pct
0,Azambuja,Azambuja,2239,80827.47,25656.463142,347220.731337,410365.638853,63144.907515,76.918926,4.295826,13.533461,15.387474
1,Azambuja,Maia,635,22854.0,7903.631288,264224.529274,311963.814516,47739.285242,85.364625,11.561413,33.430776,15.302828
2,Azambuja,Vila do Conde,545,24727.0,5842.172635,237488.457958,268051.702061,30563.244103,88.432953,9.604419,40.650709,11.401996
3,Santa Iria de Azoia,Getafe,244,8052.0,0.0,144318.405262,143891.0,-427.405262,100.0,17.923299,<NA>,-0.297034
4,Vila Nova da Rainha,Azambuja,658,18422.26,5552.491998,96427.601315,70471.397519,-25956.203795,69.73315,5.234298,17.366545,-36.832254
...,...,...,...,...,...,...,...,...,...,...,...,...
644,Vila Nova da Rainha,Paço de Arcos,1,2.0,0.02581,51.167412,0.0,-51.167412,33.333333,25.583706,1982.464619,<NA>
645,Paços de Ferreira,Vila Nova da Rainha,1,1.0,0.0,51.167412,0.0,-51.167412,<NA>,51.167412,<NA>,<NA>
646,Vila Nova da Rainha,Oliveira de Azeméis,1,32.0,6.61473,51.167412,87.012479,35.845068,<NA>,1.598982,7.735374,41.195318
647,Olhão,,1,12.0,3.25486,51.167412,0.0,-51.167412,<NA>,4.263951,15.720311,<NA>


## 09. Qualidade operacional

In [11]:
# ==========================
# 1. Estados de ocupação
# ==========================
qualidade_ocupacao = (
    viagens_comparaveis.groupby(
        "ocupacao_estado",
        dropna=False,
        as_index=False,
    )
    .agg(
        viagens=("CODEUT", "nunique"),
        paletes=("paletes", "sum"),
        peso_ton=("peso_ton", "sum"),
    )
)

# ==========================
# 2. Peso relativo
# ==========================
qualidade_ocupacao["pct_viagens"] = (
    dividir_seguro(
        qualidade_ocupacao["viagens"],
        pd.Series(
            qualidade_ocupacao[
                "viagens"
            ].sum(),
            index=qualidade_ocupacao.index,
        ),
    )
    * 100
)

qualidade_ocupacao

,ocupacao_estado,viagens,paletes,peso_ton,pct_viagens
0,EXCESSO_CAPACIDADE,2539,101883.251,24115.373661,16.188472
1,OK,12563,249391.523,63301.053064,80.100740
2,SEM_CAPACIDADE,582,19486.0,5955.301041,3.710788


## 10. Danone — reconciliação

In [12]:
# ==========================
# 1. Resumo de reconciliação
# ==========================
danone_resumo = (
    validacao_danone.copy()
)

# ==========================
# 2. Peso financeiro
# ==========================
total_danone_reconciliado = (
    danone_resumo["ingresso"]
    .sum()
)

danone_resumo["peso_pct"] = (
    dividir_seguro(
        danone_resumo["ingresso"],
        pd.Series(
            total_danone_reconciliado,
            index=danone_resumo.index,
        ),
    )
    * 100
)

danone_resumo

,estado_reconciliacao,casos,referencias,ingresso_base,extra,ingresso,peso_pct
0,ASSOCIADO_EXACTO,13866,13866,1.050288e+06,152965.109168,1.203253e+06,95.556946
1,EXCLUIDO_CODACT_13,24565,24564,4.130858e+04,5962.101594,4.727069e+04,3.754025
2,FORA_INFORM27,2165,2165,5.491476e+03,779.558457,6.271034e+03,0.498017
3,REFERENCIA_EXISTE_DATA_DIFERENTE,83,83,2.112202e+03,293.022721,2.405225e+03,0.191012


## 11. Fecho

In [13]:
# ==========================
# 1. Totais oficiais comparáveis
# ==========================
fecho = pd.DataFrame({
    "metrica": [
        "Ingresso Jan-Ago",
        "Custo Jan-Ago",
        "Margem Jan-Ago",
        "Margem % Jan-Ago",
        "Paletes Jan-Ago",
        "Peso ton Jan-Ago",
        "Viagens Jan-Ago",
        "Ocupação média Jan-Ago",
    ],
    "valor": [
        df_comparavel[
            "total_ingresso_c"
        ].sum(),
        df_comparavel[
            "custo_total_c"
        ].sum(),
        df_comparavel[
            "margem"
        ].sum(),
        (
            df_comparavel[
                "margem"
            ].sum()
            / df_comparavel[
                "total_ingresso_c"
            ].sum()
            * 100
        )
        if df_comparavel[
            "total_ingresso_c"
        ].sum() != 0
        else np.nan,
        df_comparavel[
            "PALETS"
        ].sum(),
        df_comparavel[
            "PESO_BRUTO"
        ].sum() / 1000,
        viagens_comparaveis[
            "CODEUT"
        ].nunique(),
        viagens_validas[
            "ocupacao_pct"
        ].mean(),
    ],
})

fecho

,metrica,valor
0,Ingresso Jan-Ago,4.263540e+06
1,Custo Jan-Ago,4.439654e+06
2,Margem Jan-Ago,-1.761141e+05
3,Margem % Jan-Ago,-4.130701e+00
4,Paletes Jan-Ago,3.707608e+05
5,Peso ton Jan-Ago,9.337173e+04
6,Viagens Jan-Ago,1.568400e+04
7,Ocupação média Jan-Ago,6.603686e+01
